In [ ]:
# Databricks notebook source
# 04_clean_ireland

import hashlib
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from pyspark.sql.types import DateType, DoubleType, StringType, StructField, StructType

CATALOG = "decide_catalog"
SCHEMA = "decide_schema"
VOLUME_PATH = "/Volumes/decide_catalog/decide_schema/decide_volume"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")


def should_run_pipeline():
    try:
        value = dbutils.jobs.taskValues.get(taskKey="00_check_source_changes", key="should_run", default="true")
        return str(value).lower() == "true"
    except Exception:
        return True


def source_path(file_name):
    return f"{VOLUME_PATH}/{file_name}"


def sha256_hash(value):
    if pd.isna(value):
        return None
    return hashlib.sha256(str(value).encode("utf-8")).hexdigest()


def month_start(values):
    return pd.to_datetime(values, errors="coerce").dt.to_period("M").dt.start_time.dt.date


def week_start(values, week_start="sunday"):
    dates = pd.to_datetime(values, errors="coerce")
    freq = "W-SAT" if week_start == "sunday" else "W-SUN"
    return dates.dt.to_period(freq).dt.start_time.dt.date


def max_with_na(series):
    values = pd.to_numeric(series, errors="coerce")
    if values.notna().any():
        return values.max(skipna=True)
    return np.nan


def write_delta(pdf, table_name):
    output_cols = [
        "Lab_reference",
        "Country",
        "Breed",
        "Province",
        "Farm_ID",
        "Diagnostic_test",
        "Sample_type",
        "Samplenumber",
        "Date",
        "Date_month",
        "Date_week",
        "Pathogen",
        "Result",
    ]
    pdf = pdf.reindex(columns=output_cols).copy()

    string_cols = [
        "Lab_reference",
        "Country",
        "Breed",
        "Province",
        "Farm_ID",
        "Diagnostic_test",
        "Sample_type",
        "Samplenumber",
        "Pathogen",
    ]
    for col in string_cols:
        pdf[col] = pdf[col].where(pd.notna(pdf[col]), None).astype(object)
    for col in ["Date", "Date_month", "Date_week"]:
        pdf[col] = pd.to_datetime(pdf[col], errors="coerce").dt.date
    pdf["Result"] = pd.to_numeric(pdf["Result"], errors="coerce")

    schema = StructType(
        [
            StructField("Lab_reference", StringType(), True),
            StructField("Country", StringType(), True),
            StructField("Breed", StringType(), True),
            StructField("Province", StringType(), True),
            StructField("Farm_ID", StringType(), True),
            StructField("Diagnostic_test", StringType(), True),
            StructField("Sample_type", StringType(), True),
            StructField("Samplenumber", StringType(), True),
            StructField("Date", DateType(), True),
            StructField("Date_month", DateType(), True),
            StructField("Date_week", DateType(), True),
            StructField("Pathogen", StringType(), True),
            StructField("Result", DoubleType(), True),
        ]
    )
    sdf = spark.createDataFrame(pdf, schema=schema)
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    (
        sdf.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_name)
    )
    print(f"Wrote {sdf.count()} rows to {full_name}")

if not should_run_pipeline():
    dbutils.notebook.exit("No source changes detected; skipping Ireland")

files = [
    "Jade_2021_Final_Anonymised_data_Only_2023-04-20.v2.xlsx",
    "Jade_2022_Final_Anonymised_data_Only_2023-04-21.xlsx",
    "Final_Anonymised_data_Only_2023_2025-03-04.xlsx",
]
raw = pd.concat([pd.read_excel(source_path(name), engine="openpyxl") for name in files], ignore_index=True)
allowed_matrix = ["Pleural Fluid", "Tissue swab", "Tonsil", "Lymph Node - Multiple", "Trachea", "Thoracic Fluid", "Lung", "Swab", "Culture", "Thymus", "Part Carcass", "Nasal Swab", "Nasal Fluid", "Tissue-Pool", "Tissue (VTM)", "Carcass", "Lymph Node", "Pooled swab", "Misc."]
allowed_tests = ["PI3V PCR", "PCR M. haemolytica - ARVL", "Mycoplasma bovis (PCR)", "PCR H. somni - ARVL", "PCR P. multocida - ARVL", "Miscellaneous Test", "Routine Culture", "PCR M. bovis - ARVL", "BRSV PCR", "Culture Growth", "PCR BoCoV", "Mycoplasma bovis (PCR)"]
df = raw[(raw["SYSTEM"].isin(["Respiratory", "NA"])) & (raw["ALIQUOTMATRIXTYPE"].isin(allowed_matrix)) & (raw["TEST"].isin(allowed_tests))].copy()
df = df.rename(columns={"SDGa": "Filenumber", "SAMPLEa": "Samplenumber", "HERD_NOa": "Farm_ID", "DELIVERY_DATE": "Date", "Herd.Type": "breed"})
df["Country"] = "Ireland"
df["Lab_reference"] = "5"
df["Sample_type"] = np.select([df["SUBCLASS"].eq("Carcass"), df["ALIQUOTMATRIXTYPE"].isin(["Carcass", "Lung", "Thymus", "Lymph Node - Multiple", "Tissue-Pool", "Lymph Node", "Tissue (VTM)", "Part Carcass"]), df["ALIQUOTMATRIXTYPE"].isin(["Swab", "Nasal Swab", "Pooled swab", "Nasal Fluid"]), df["ALIQUOTMATRIXTYPE"].isin(["Trachea", "Thoracic Fluid", "Culture", "Fluid", "Misc.", "Pleural Fluid"])], ["Autopsy", "Autopsy", "Swab", "Unknown"], default="Missing")
df["Diagnostic_test"] = np.select([df["TEST"].isin(["PI3V PCR", "PCR M. haemolytica - ARVL", "Mycoplasma bovis (PCR)", "PCR H. somni - ARVL", "PCR M. bovis - ARVL", "BRSV PCR", "PCR BoCoV", "PCR P. multocida - ARVL"]), df["TEST"].isin(["Routine Culture", "Culture Growth"])], ["PCR", "Culture"], default="Missing")
df["Breed"] = df["breed"].map({"BEEF": "Beef", "DAIRY": "Dairy", "SUCKLER": "Suckler", "OTHER": "Unknown"}).fillna("Unknown")
df["Province"] = df["County"]
df["Pathogen"] = df["TEST"].map({"PCR P. multocida - ARVL": "PM", "PCR M. haemolytica - ARVL": "MH", "PCR H. somni - ARVL": "HS", "H. somni PCR": "HS", "Mycoplasma bovis (PCR)": "MB", "PCR M. bovis - ARVL": "MB", "PI3V PCR": "PI3", "PCR BoCoV": "BCV", "BRSV PCR": "BRSV"}).fillna("Missing")
barometer_dt = df[["Filenumber", "Samplenumber", "Diagnostic_test", "Country", "Lab_reference", "Sample_type", "Breed", "Pathogen", "Date", "Province", "RESULT", "RESULTNAME", "AGENT", "Farm_ID"]].drop_duplicates()
for col in ["Filenumber", "Samplenumber", "Farm_ID"]:
    barometer_dt[col] = barometer_dt[col].apply(sha256_hash)
for p in ["HS", "MH", "PM"]:
    barometer_dt[p] = np.where(barometer_dt["Diagnostic_test"].eq("Culture"), 0, np.nan)
long = barometer_dt.melt(id_vars=["Filenumber", "Samplenumber", "Diagnostic_test", "Country", "Lab_reference", "Sample_type", "Breed", "Pathogen", "Date", "Province", "RESULT", "RESULTNAME", "AGENT", "Farm_ID"], value_vars=["PM", "MH", "HS"], var_name="Pathogen_culture", value_name="Result_culture")
long.loc[long["Pathogen"].eq("Missing"), "Pathogen"] = long["Pathogen_culture"]

def translate_result(row):
    result = row["RESULT"]
    if row["Diagnostic_test"] == "PCR":
        if result in ["Positive", "Weak Positive", "Mycoplasma bovis PCR Positive", "Strong Positive"]:
            return 1
        if result in ["No Pathogen detected", "Negative", "Sterile", "No Significant Growth", "No CT", "Mycoplasma bovis PCR Negative", "Mixed Non-Significant Bacterial Growth", "No Significant Growth @48hrs", "No Growth", "No Pathogen detectedn", "No RNA detected", "No DNA detected", "No Virus Detected", "Not Detected"]:
            return 0
        if result in ["Inconclusive", "Mixed Bacterial Growth", "Mixed Growth", "Very Mixed Growth"]:
            return np.nan
    if row["Diagnostic_test"] == "Culture" and row["Pathogen"] in ["MH", "PM", "HS"]:
        if row["Pathogen"] == "MH" and result == "Mannheimia haemolytica":
            return 1
        if row["Pathogen"] == "PM" and result in ["Pasteurella multocida", "P. multocida"]:
            return 1
        if row["Pathogen"] == "HS" and result in ["Histophilus somni", "Histophilus somnus", "Histophilus somnii"]:
            return 1
        return 0
    return np.nan

long["Result"] = long.apply(translate_result, axis=1)
long["Date"] = pd.to_datetime(long["Date"], errors="coerce")
long["Date_month"] = month_start(long["Date"])
long["Date_week"] = week_start(long["Date"], week_start="sunday")
group_cols = ["Lab_reference", "Country", "Breed", "Province", "Farm_ID", "Diagnostic_test", "Sample_type", "Pathogen", "Samplenumber", "Date_month", "Date_week", "Date"]
barometer = long.groupby(group_cols, dropna=False)["Result"].agg(max_with_na).reset_index()
barometer = barometer[["Lab_reference", "Country", "Breed", "Province", "Farm_ID", "Diagnostic_test", "Sample_type", "Samplenumber", "Date", "Date_month", "Date_week", "Pathogen", "Result"]]
write_delta(barometer, "barometer_ireland")
